In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

# Load the original dataset
data = pd.read_csv('creditcard.csv')

# Step 1: Generate a unique identifier for each transaction
data['customer_id'] = range(1, len(data) + 1)

# Step 2: Generate a synthetic phone number for each transaction
def generate_phone_number():
    return f"+1-{np.random.randint(100, 999)}-{np.random.randint(100, 999)}-{np.random.randint(1000, 9999)}"

data['phone_number'] = [generate_phone_number() for _ in range(len(data))]

# Step 3: Standardize the 'Time' and 'Amount' features
scaler = StandardScaler()
data['Time'] = scaler.fit_transform(data[['Time']])
data['Amount'] = scaler.fit_transform(data[['Amount']])

# Step 4: Separate features and target variable
X = data.drop('Class', axis=1)  # Features
y = data['Class']  # Target variable

# Step 5: Combine features and target for balancing
train_data = pd.concat([X, y], axis=1)

# Step 6: Separate the majority and minority classes
non_fraud = train_data[train_data['Class'] == 0]
fraud = train_data[train_data['Class'] == 1]

# Step 7: Balance the dataset by undersampling the majority class
non_fraud_sampled = non_fraud.sample(len(fraud), random_state=42)

# Step 8: Combine the balanced dataset
balanced_data = pd.concat([non_fraud_sampled, fraud])

# Step 9: Shuffle the balanced dataset
balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)


# # Step 11: Save the balanced dataset with identifiers for Kafka ingestion
# balanced_data.to_csv('balanced_data.csv', index=False)
# print("Balanced dataset with identifiers saved to 'balanced_data.csv' for Kafka ingestion.")


In [2]:
balanced_data = balanced_data.drop(columns=['customer_id', 'phone_number'])
X_balanced = balanced_data.drop('Class', axis=1)
y_balanced = balanced_data['Class']
X_train, X_val, y_train, y_val = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42)

# Step 13: Check the numbers of our data
print("Training data distribution:")
print(y_train.value_counts())
print("Validation data distribution:")
print(y_val.value_counts())

Training data distribution:
Class
1    405
0    382
Name: count, dtype: int64
Validation data distribution:
Class
0    110
1     87
Name: count, dtype: int64


In [3]:
# Step 14: Discover the best class weights
class_weights = [{0: 1, 1: 1}, {0: 1, 1: 0.1}, {0: 1, 1: 0.01}, {0: 1, 1: 0.001}]
best_accuracy = 0
best_weights = None

for weights in class_weights:
    model = LogisticRegression(class_weight=weights)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_weights = weights

print(f"Best class weights: {best_weights}")

Best class weights: {0: 1, 1: 1}


In [5]:
# Step 15: Find the best penalty strength (C)

from sklearn.model_selection import GridSearchCV


param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
model = LogisticRegression(class_weight=best_weights)
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

best_C = grid_search.best_params_['C']
print(f"Best penalty strength (C): {best_C}")

Best penalty strength (C): 100


In [8]:
#  Step 16: Train the final model with the best class weights and penalty strength
final_model = LogisticRegression(class_weight=best_weights, C=best_C)
final_model.fit(X_train, y_train)

# Step 17: Evaluate the model on the validation set
y_pred = final_model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy on validation set: {accuracy:.2f}")

# Step 18: Save the trained model
joblib.dump(final_model, 'fraud_model.pkl')
print("Trained model saved to 'fraud_detection_model.pkl'")

Accuracy on validation set: 0.93
Trained model saved to 'fraud_detection_model.pkl'


In [9]:

# Step 19: Extract coefficients for use in Flink
coefficients = final_model.coef_[0]
intercept = final_model.intercept_[0]
print("Model coefficients:", coefficients)
print("Model intercept:", intercept)

# You can now use the coefficients and intercept in your Flink application.

Model coefficients: [-0.7053428   1.52027868  1.49609711  0.38911819  1.05612273  1.28066138
 -1.08789901 -1.9726092  -1.02467475 -1.11613632 -1.77191102  1.53542393
 -2.78634724 -0.69264159 -3.95581356 -0.2674069  -2.39490204 -3.95983395
 -1.13627911  0.99180146 -1.80016766 -0.14902391  1.096772    1.35900139
 -0.12291551 -0.32259655  0.0062771  -0.78077786  4.54260742  4.63266889]
Model intercept: -4.5011820850563655
